# MIMIC-IV Sepsis-3 Table Creation for BigQuery

This notebook creates all necessary tables for Sepsis-3 analysis in BigQuery.

**Target Dataset:** `my-new-project-473015.my_mimiciv_derived`  
**Region:** US  
**Source:** MIMIC-IV v3.1 from PhysioNet

## Dependencies Order:
1. vitalsign (base table)
2. weight_durations (base table)
3. urine_output (base table)
4. urine_output_rate (depends on urine_output, weight_durations)
5. ventilator_setting (base table - prerequisite for ventilation)
6. ventilation (depends on ventilator_setting, oxygen_delivery)
7. suspicion_of_infection (depends on antibiotic, microbiologyevents)
8. sofa (depends on vitalsign, ventilation, urine_output_rate, and multiple derived tables)
9. sepsis3 (depends on suspicion_of_infection, sofa)

---

## Setup

First, we'll import necessary libraries and create our dataset.

**YOU SHOULD CHANGE YOUR PROJECT AND DATASET NAME**

In [37]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

# Initialize BigQuery client
client = bigquery.Client(project='my-new-project-473015')

# Dataset configuration
DATASET_ID = 'my_mimiciv_derived'
PROJECT_ID = 'my-new-project-473015'

print("="*70)
print(f"MIMIC-IV Sepsis-3 Table Creation")
print(f"Target dataset: {PROJECT_ID}.{DATASET_ID}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

# Create dataset if it doesn't exist
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
try:
    client.get_dataset(dataset_ref)
    print(f"\n✓ Dataset {DATASET_ID} already exists")
except:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"  # Same region as MIMIC-IV
    client.create_dataset(dataset)
    print(f"\n✓ Created dataset {DATASET_ID} in US region")

print("\n✅ Setup complete! Ready to create tables.")

MIMIC-IV Sepsis-3 Table Creation
Target dataset: my-new-project-473015.my_mimiciv_derived
Timestamp: 2025-10-09 15:14:43

✓ Dataset my_mimiciv_derived already exists

✅ Setup complete! Ready to create tables.


---

## STEP 1: Create Vitalsign Table

This table extracts and pivots vital signs data from ICU chartevents.

**Contains:**
- Heart rate, blood pressure (systolic/diastolic/mean)
- Respiratory rate
- SpO2 (oxygen saturation)
- Temperature (converted to Celsius)
- Glucose levels

**Source:** `chartevents` table  
**One row per:** patient charttime  
URL: mimic-iv/concepts/measurement/vitalsign.sql

In [7]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.vitalsign` AS
SELECT
    ce.subject_id
    , ce.stay_id
    , ce.charttime
    , AVG(CASE WHEN itemid IN (220045)
            AND valuenum > 0
            AND valuenum < 300
            THEN valuenum END
    ) AS heart_rate
    , AVG(CASE WHEN itemid IN (220179, 220050, 225309)
            AND valuenum > 0
            AND valuenum < 400
            THEN valuenum END
    ) AS sbp
    , AVG(CASE WHEN itemid IN (220180, 220051, 225310)
                AND valuenum > 0
                AND valuenum < 300
                THEN valuenum END
    ) AS dbp
    , AVG(CASE WHEN itemid IN (220052, 220181, 225312)
                AND valuenum > 0
                AND valuenum < 300
                THEN valuenum END
    ) AS mbp
    , AVG(CASE WHEN itemid = 220179
                AND valuenum > 0
                AND valuenum < 400
                THEN valuenum END
    ) AS sbp_ni
    , AVG(CASE WHEN itemid = 220180
                AND valuenum > 0
                AND valuenum < 300
                THEN valuenum END
    ) AS dbp_ni
    , AVG(CASE WHEN itemid = 220181
                AND valuenum > 0
                AND valuenum < 300
                THEN valuenum END
    ) AS mbp_ni
    , AVG(CASE WHEN itemid IN (220210, 224690)
                AND valuenum > 0
                AND valuenum < 70
                THEN valuenum END
    ) AS resp_rate
    , ROUND(CAST(
            AVG(CASE
                WHEN itemid IN (223761)
                    AND valuenum > 70
                    AND valuenum < 120
                    THEN (valuenum - 32) / 1.8
                WHEN itemid IN (223762)
                    AND valuenum > 10
                    AND valuenum < 50
                    THEN valuenum END)
            AS NUMERIC), 2) AS temperature
    , MAX(CASE WHEN itemid = 224642 THEN value END
    ) AS temperature_site
    , AVG(CASE WHEN itemid IN (220277)
                AND valuenum > 0
                AND valuenum <= 100
                THEN valuenum END
    ) AS spo2
    , AVG(CASE WHEN itemid IN (225664, 220621, 226537)
                AND valuenum > 0
                THEN valuenum END
    ) AS glucose
FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
WHERE ce.stay_id IS NOT NULL
    AND ce.itemid IN
    (
        220045, 225309, 225310, 225312, 220050, 220051, 220052,
        220179, 220180, 220181, 220210, 224690, 220277,
        225664, 220621, 226537, 223762, 223761, 224642
    )
GROUP BY ce.subject_id, ce.stay_id, ce.charttime;

Query is running:   0%|          |

""


In [8]:
%%bigquery

SELECT
    'vitalsign' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    COUNT(DISTINCT subject_id) as unique_patients
FROM `my-new-project-473015.my_mimiciv_derived.vitalsign`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,unique_patients
0,vitalsign,13519533,94446,65366


---

## STEP 2: Create Weight Durations Table

This table extracts patient weight measurements with start/stop times for each weight period.

**Contains:**
- Admission weights and daily weights
- Start and end times for each weight measurement period
- Weight type (admit vs daily)

**Purpose:** Used to calculate weight-adjusted metrics (e.g., urine output per kg)

**Source:** `chartevents` table (itemids: 226512=Admit Weight, 224639=Daily Weight)  
URL:mimic-iv/concepts/demographics/weight_durations.sql

In [9]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.weight_durations` AS
WITH wt_stg AS (
    SELECT
        c.stay_id
        , c.charttime
        , CASE WHEN c.itemid = 226512 THEN 'admit'
            ELSE 'daily' END AS weight_type
        , c.valuenum AS weight
    FROM `physionet-data.mimiciv_3_1_icu.chartevents` c
    WHERE c.valuenum IS NOT NULL
        AND c.itemid IN (226512, 224639)
        AND c.valuenum > 0
)
, wt_stg1 AS (
    SELECT
        stay_id, charttime, weight_type, weight
        , ROW_NUMBER() OVER (
            PARTITION BY stay_id, weight_type ORDER BY charttime
        ) AS rn
    FROM wt_stg
    WHERE weight IS NOT NULL
)
, wt_stg2 AS (
    SELECT
        wt_stg1.stay_id, ie.intime, ie.outtime, wt_stg1.weight_type
        , CASE WHEN wt_stg1.weight_type = 'admit' AND wt_stg1.rn = 1
            THEN DATETIME_SUB(ie.intime, INTERVAL '2' HOUR)
            ELSE wt_stg1.charttime END AS starttime
        , wt_stg1.weight
    FROM wt_stg1
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON ie.stay_id = wt_stg1.stay_id
)
, wt_stg3 AS (
    SELECT
        stay_id, intime, outtime, starttime
        , COALESCE(
            LEAD(starttime) OVER (PARTITION BY stay_id ORDER BY starttime)
            , DATETIME_ADD(outtime, INTERVAL '2' HOUR)
        ) AS endtime
        , weight, weight_type
    FROM wt_stg2
)
, wt1 AS (
    SELECT
        stay_id, starttime
        , COALESCE(
            endtime
            , LEAD(starttime) OVER (PARTITION BY stay_id ORDER BY starttime)
            , DATETIME_ADD(outtime, INTERVAL '2' HOUR)
        ) AS endtime
        , weight, weight_type
    FROM wt_stg3
)
, wt_fix AS (
    SELECT ie.stay_id
        , DATETIME_SUB(ie.intime, INTERVAL '2' HOUR) AS starttime
        , wt.starttime AS endtime
        , wt.weight, wt.weight_type
    FROM `physionet-data.mimiciv_3_1_icu.icustays` ie
    INNER JOIN (
            SELECT wt1.stay_id, wt1.starttime, wt1.weight, weight_type
                , ROW_NUMBER() OVER (
                    PARTITION BY wt1.stay_id ORDER BY wt1.starttime
                ) AS rn
            FROM wt1
        ) wt
        ON ie.stay_id = wt.stay_id
            AND wt.rn = 1
            AND ie.intime < wt.starttime
)
SELECT stay_id, starttime, endtime, weight, weight_type FROM wt1
UNION ALL
SELECT stay_id, starttime, endtime, weight, weight_type FROM wt_fix;

Query is running:   0%|          |

""


In [10]:
%%bigquery

SELECT
    'weight_durations' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    ROUND(AVG(weight), 1) as avg_weight_kg,
    MIN(weight) as min_weight,
    MAX(weight) as max_weight
FROM `my-new-project-473015.my_mimiciv_derived.weight_durations`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_weight_kg,min_weight,max_weight
0,weight_durations,401850,91163,90.5,0.5,1103030.0


-- TODO: eliminate obvious outliers if there is a reasonable weight
We deal with it in "urine_output_rate"


---

## STEP 3: Create Urine Output Table

This table aggregates urine output volumes from various collection sources.

**Contains:**
- Total urine output per charttime
- Includes various collection methods (Foley catheter, void, nephrostomy, etc.)
- GU irrigant volume is subtracted (negative value)

**Source:** `outputevents` table  
**One row per:** stay_id and charttime  
URL: mimic-iv/concepts/measurement/urine_output.sql

In [11]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.urine_output` AS
WITH uo AS (
    SELECT
        oe.stay_id
        , oe.charttime
        , CASE
            WHEN oe.itemid = 227488 AND oe.value > 0 THEN -1 * oe.value
            ELSE oe.value
        END AS urineoutput
    FROM `physionet-data.mimiciv_3_1_icu.outputevents` oe
    WHERE itemid IN (
            226559, 226560, 226561, 226584, 226563, 226564, 226565,
            226567, 226557, 226558, 227488, 227489
        )
)
SELECT
    stay_id
    , charttime
    , SUM(urineoutput) AS urineoutput
FROM uo
GROUP BY stay_id, charttime;

Query is running:   0%|          |

""


In [12]:
%%bigquery

SELECT
    'urine_output' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    ROUND(AVG(urineoutput), 1) as avg_uo_ml,
    MIN(urineoutput) as min_uo,
    MAX(urineoutput) as max_uo
FROM `my-new-project-473015.my_mimiciv_derived.urine_output`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_uo_ml,min_uo,max_uo
0,urine_output,4127634,90471,145.1,-6000.0,876587.0


itemid 227488 = GU Irrigant Volume In (input for urinary irrigant)
This is intentionally negative
Reason: because irrigant is a fluid you put into your body, so you need to subtract it from the actual urine volume
The comment also says "GU irrigant volume in usually has a corresponding volume out, so the net is often 0".

---

## STEP 4: Create Urine Output Rate Table

This table calculates urine output rates per hour over different time windows.

**Contains:**
- Urine output over 6, 12, and 24 hour windows
- Weight-adjusted urine output (ml/kg/hr)
- Time duration for each measurement window

**Purpose:** Used for SOFA renal component calculation (oliguria detection)

**Dependencies:** `urine_output` and `weight_durations` tables  
**Note:** This query may take a few minutes due to self-join operations  
URL: mimic-iv/concepts/measurement/urine_output_rate.sql

In [15]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.urine_output_rate` AS
WITH tm AS (
    SELECT ie.stay_id
           , MIN(charttime) AS intime_hr
           , MAX(charttime) AS outtime_hr
    FROM `physionet-data.mimiciv_3_1_icu.icustays` ie
    INNER JOIN `physionet-data.mimiciv_3_1_icu.chartevents` ce
        ON ie.stay_id = ce.stay_id
            AND ce.itemid = 220045
            AND ce.charttime > DATETIME_SUB(ie.intime, INTERVAL '1' MONTH)
            AND ce.charttime < DATETIME_ADD(ie.outtime, INTERVAL '1' MONTH)
    GROUP BY ie.stay_id
)
, uo_tm AS (
    SELECT tm.stay_id
        , CASE
            WHEN LAG(charttime) OVER w IS NULL
                THEN DATETIME_DIFF(charttime, intime_hr, MINUTE)
            ELSE DATETIME_DIFF(charttime, LAG(charttime) OVER w, MINUTE)
        END AS tm_since_last_uo
        , uo.charttime
        , uo.urineoutput
    FROM tm
    INNER JOIN `my-new-project-473015.my_mimiciv_derived.urine_output` uo
        ON tm.stay_id = uo.stay_id
    WINDOW w AS (PARTITION BY tm.stay_id ORDER BY charttime)
)
, ur_stg AS (
    SELECT io.stay_id, io.charttime
        , SUM(DISTINCT io.urineoutput) AS uo
        , SUM(CASE WHEN DATETIME_DIFF(io.charttime, iosum.charttime, HOUR) <= 5
            THEN iosum.urineoutput
            ELSE null END) AS urineoutput_6hr
        , SUM(CASE WHEN DATETIME_DIFF(io.charttime, iosum.charttime, HOUR) <= 5
            THEN iosum.tm_since_last_uo
            ELSE null END) / 60.0 AS uo_tm_6hr
        , SUM(CASE WHEN DATETIME_DIFF(io.charttime, iosum.charttime, HOUR) <= 11
            THEN iosum.urineoutput
            ELSE null END) AS urineoutput_12hr
        , SUM(CASE WHEN DATETIME_DIFF(io.charttime, iosum.charttime, HOUR) <= 11
            THEN iosum.tm_since_last_uo
            ELSE null END) / 60.0 AS uo_tm_12hr
        , SUM(iosum.urineoutput) AS urineoutput_24hr
        , SUM(iosum.tm_since_last_uo) / 60.0 AS uo_tm_24hr
    FROM uo_tm io
    LEFT JOIN uo_tm iosum
        ON io.stay_id = iosum.stay_id
            AND io.charttime >= iosum.charttime
            AND io.charttime <= DATETIME_ADD(iosum.charttime, INTERVAL '23' HOUR)
    GROUP BY io.stay_id, io.charttime
)
SELECT
    ur.stay_id
    , ur.charttime
    , wd.weight
    , ur.uo
    , ur.urineoutput_6hr
    , ur.urineoutput_12hr
    , ur.urineoutput_24hr
    , CASE
        WHEN uo_tm_6hr >= 6 THEN ROUND(
            CAST((ur.urineoutput_6hr / wd.weight / uo_tm_6hr) AS NUMERIC), 4
        )
    END AS uo_mlkghr_6hr
    , CASE
        WHEN uo_tm_12hr >= 12 THEN ROUND(
            CAST((ur.urineoutput_12hr / wd.weight / uo_tm_12hr) AS NUMERIC), 4
        )
    END AS uo_mlkghr_12hr
    , CASE
        WHEN uo_tm_24hr >= 24 THEN ROUND(
            CAST((ur.urineoutput_24hr / wd.weight / uo_tm_24hr) AS NUMERIC), 4
        )
    END AS uo_mlkghr_24hr
    , ROUND(CAST(uo_tm_6hr AS NUMERIC), 2) AS uo_tm_6hr
    , ROUND(CAST(uo_tm_12hr AS NUMERIC), 2) AS uo_tm_12hr
    , ROUND(CAST(uo_tm_24hr AS NUMERIC), 2) AS uo_tm_24hr
FROM ur_stg ur
LEFT JOIN `my-new-project-473015.my_mimiciv_derived.weight_durations` wd
    ON ur.stay_id = wd.stay_id
        AND ur.charttime > wd.starttime
        AND ur.charttime <= wd.endtime
        AND wd.weight > 0;

Query is running:   0%|          |

""


In [16]:
%%bigquery

SELECT
    'urine_output_rate' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    ROUND(AVG(urineoutput_24hr), 1) as avg_24hr_uo_ml,
    ROUND(AVG(uo_mlkghr_24hr), 2) as avg_uo_rate_mlkghr
FROM `my-new-project-473015.my_mimiciv_derived.urine_output_rate`
WHERE uo_mlkghr_24hr IS NOT NULL;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_24hr_uo_ml,avg_uo_rate_mlkghr
0,urine_output_rate,2648691,62687,2251.5,1.140000000


---

## STEP 5a: Create Ventilator Setting Table (Prerequisite)

This table extracts ventilator settings from chartevents.

**Contains:**
- Respiratory rate (set, total, spontaneous)
- Tidal volume settings
- PEEP, FiO2, plateau pressure
- Ventilator mode and type

**Source:** `chartevents` table  
**Note:** This is a prerequisite for the ventilation table  
URL; mimic-iv/concepts/measurement/ventilator_setting.sql

In [19]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.ventilator_setting` AS
WITH ce AS (
    SELECT
        ce.subject_id
        , ce.stay_id
        , ce.charttime
        , itemid
        , value
        , CASE
            WHEN itemid = 223835
                THEN
                CASE
                    WHEN valuenum >= 0.20 AND valuenum <= 1
                        THEN valuenum * 100
                    WHEN valuenum > 1 AND valuenum < 20
                        THEN null
                    WHEN valuenum >= 20 AND valuenum <= 100
                        THEN valuenum
                    ELSE null END
            WHEN itemid IN (220339, 224700)
                THEN
                CASE
                    WHEN valuenum > 100 THEN null
                    WHEN valuenum < 0 THEN null
                    ELSE valuenum END
            ELSE valuenum END AS valuenum
        , valueuom
        , storetime
    FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
    WHERE ce.value IS NOT NULL
        AND ce.stay_id IS NOT NULL
        AND ce.itemid IN
        (
            224688, 224689, 224690, 224687, 224685, 224684, 224686,
            224696, 220339, 224700, 223835, 223849, 229314, 223848, 224691
        )
)
SELECT
    subject_id
    , MAX(stay_id) AS stay_id
    , charttime
    , MAX(CASE WHEN itemid = 224688 THEN valuenum ELSE null END) AS respiratory_rate_set
    , MAX(CASE WHEN itemid = 224690 THEN valuenum ELSE null END) AS respiratory_rate_total
    , MAX(CASE WHEN itemid = 224689 THEN valuenum ELSE null END) AS respiratory_rate_spontaneous
    , MAX(CASE WHEN itemid = 224687 THEN valuenum ELSE null END) AS minute_volume
    , MAX(CASE WHEN itemid = 224684 THEN valuenum ELSE null END) AS tidal_volume_set
    , MAX(CASE WHEN itemid = 224685 THEN valuenum ELSE null END) AS tidal_volume_observed
    , MAX(CASE WHEN itemid = 224686 THEN valuenum ELSE null END) AS tidal_volume_spontaneous
    , MAX(CASE WHEN itemid = 224696 THEN valuenum ELSE null END) AS plateau_pressure
    , MAX(CASE WHEN itemid IN (220339, 224700) THEN valuenum ELSE null END) AS peep
    , MAX(CASE WHEN itemid = 223835 THEN valuenum ELSE null END) AS fio2
    , MAX(CASE WHEN itemid = 224691 THEN valuenum ELSE null END) AS flow_rate
    , MAX(CASE WHEN itemid = 223849 THEN value ELSE null END) AS ventilator_mode
    , MAX(CASE WHEN itemid = 229314 THEN value ELSE null END) AS ventilator_mode_hamilton
    , MAX(CASE WHEN itemid = 223848 THEN value ELSE null END) AS ventilator_type
FROM ce
GROUP BY subject_id, charttime;

Query is running:   0%|          |

""


In [20]:
%%bigquery

SELECT
    'ventilator_setting' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays
FROM `my-new-project-473015.my_mimiciv_derived.ventilator_setting`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays
0,ventilator_setting,1377514,47445


---

## STEP 5b: Create Ventilation Table

This table classifies oxygen delivery devices and ventilator modes into six clinical categories.

**Categories:**
1. **Tracheostomy** - Tracheostomy tube with/without ventilation
2. **InvasiveVent** - Invasive positive pressure ventilation via endotracheal tube
3. **NonInvasiveVent** - Non-invasive positive pressure (BiPAP, CPAP)
4. **HFNC** - High flow nasal cannula
5. **SupplementalOxygen** - Other oxygen delivery (nasal cannula, face mask, etc.)
6. **None** - No oxygen support

**Priority:** trach > invasive vent > NIV > HFNC > supplemental O2 > none

**Dependencies:** `ventilator_setting` (just created) and `oxygen_delivery` tables

URL: mimic-iv/concepts/treatment/ventilation.sql

In [24]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.ventilation` AS
WITH tm AS (
    SELECT stay_id, charttime
    FROM `my-new-project-473015.my_mimiciv_derived.ventilator_setting`
    UNION DISTINCT
    SELECT stay_id, charttime
    FROM `physionet-data.mimiciv_3_1_derived.oxygen_delivery`
)
, vs AS (
    SELECT tm.stay_id, tm.charttime
        , o2_delivery_device_1
        , COALESCE(ventilator_mode, ventilator_mode_hamilton) AS vent_mode
        , CASE
            WHEN o2_delivery_device_1 IN ('Tracheostomy tube', 'Trach mask ')
                THEN 'Tracheostomy'
            WHEN o2_delivery_device_1 IN ('Endotracheal tube')
                OR ventilator_mode IN (
                    '(S) CMV', 'APRV', 'APRV/Biphasic+ApnPress', 'APRV/Biphasic+ApnVol',
                    'APV (cmv)', 'Ambient', 'Apnea Ventilation', 'CMV', 'CMV/ASSIST',
                    'CMV/ASSIST/AutoFlow', 'CMV/AutoFlow', 'CPAP/PPS', 'CPAP/PSV',
                    'CPAP/PSV+Apn TCPL', 'CPAP/PSV+ApnPres', 'CPAP/PSV+ApnVol',
                    'MMV', 'MMV/AutoFlow', 'MMV/PSV', 'MMV/PSV/AutoFlow', 'P-CMV',
                    'PCV+', 'PCV+/PSV', 'PCV+Assist', 'PRES/AC', 'PRVC/AC', 'PRVC/SIMV',
                    'PSV/SBT', 'SIMV', 'SIMV/AutoFlow', 'SIMV/PRES', 'SIMV/PSV',
                    'SIMV/PSV/AutoFlow', 'SIMV/VOL', 'SYNCHRON MASTER', 'SYNCHRON SLAVE',
                    'VOL/AC'
                )
                OR ventilator_mode_hamilton IN (
                    'APRV', 'APV (cmv)', 'Ambient', '(S) CMV', 'P-CMV', 'SIMV',
                    'APV (simv)', 'P-SIMV', 'VS', 'ASV'
                )
                THEN 'InvasiveVent'
            WHEN o2_delivery_device_1 IN ('Bipap mask ', 'CPAP mask ')
                OR ventilator_mode_hamilton IN ('DuoPaP', 'NIV', 'NIV-ST')
                THEN 'NonInvasiveVent'
            WHEN o2_delivery_device_1 IN ('High flow nasal cannula')
                THEN 'HFNC'
            WHEN o2_delivery_device_1 IN (
                    'Non-rebreather', 'Face tent', 'Aerosol-cool', 'Venti mask ',
                    'Medium conc mask ', 'Ultrasonic neb', 'Vapomist', 'Oxymizer',
                    'High flow neb', 'Nasal cannula'
                )
                THEN 'SupplementalOxygen'
            WHEN o2_delivery_device_1 IN ('None')
                THEN 'None'
            ELSE NULL END AS ventilation_status
    FROM tm
    LEFT JOIN `my-new-project-473015.my_mimiciv_derived.ventilator_setting` vs
        ON tm.stay_id = vs.stay_id AND tm.charttime = vs.charttime
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.oxygen_delivery` od
        ON tm.stay_id = od.stay_id AND tm.charttime = od.charttime
)
, vd0 AS (
    SELECT
        stay_id, charttime
        , LAG(charttime, 1) OVER (
            PARTITION BY stay_id, ventilation_status ORDER BY charttime
        ) AS charttime_lag
        , LEAD(charttime, 1) OVER w AS charttime_lead
        , ventilation_status
        , LAG(ventilation_status, 1) OVER w AS ventilation_status_lag
    FROM vs
    WHERE ventilation_status IS NOT NULL
    WINDOW w AS (PARTITION BY stay_id ORDER BY charttime)
)
, vd1 AS (
    SELECT
        stay_id, charttime, charttime_lag, charttime_lead, ventilation_status
        , DATETIME_DIFF(charttime, charttime_lag, MINUTE) / 60 AS ventduration
        , CASE
            WHEN ventilation_status_lag IS NULL THEN 1
            WHEN DATETIME_DIFF(charttime, charttime_lag, HOUR) >= 14 THEN 1
            WHEN ventilation_status_lag != ventilation_status THEN 1
            ELSE 0
        END AS new_ventilation_event
    FROM vd0
)
, vd2 AS (
    SELECT vd1.stay_id, vd1.charttime, vd1.charttime_lead, vd1.ventilation_status
        , ventduration, new_ventilation_event
        , SUM(new_ventilation_event) OVER (
            PARTITION BY stay_id ORDER BY charttime
        ) AS vent_seq
    FROM vd1
)
SELECT
    stay_id
    , MIN(charttime) AS starttime
    , MAX(
        CASE
            WHEN charttime_lead IS NULL
                OR DATETIME_DIFF(charttime_lead, charttime, HOUR) >= 14
                THEN charttime
            ELSE charttime_lead
        END
    ) AS endtime
    , MAX(ventilation_status) AS ventilation_status
FROM vd2
GROUP BY stay_id, vent_seq
HAVING MIN(charttime) != MAX(charttime);

Query is running:   0%|          |

""


In [25]:
%%bigquery

SELECT
    'ventilation' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    ventilation_status,
    COUNT(*) as count_per_status
FROM `my-new-project-473015.my_mimiciv_derived.ventilation`
GROUP BY ventilation_status
ORDER BY count_per_status DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,ventilation_status,count_per_status
0,ventilation,83826,60358,SupplementalOxygen,83826
1,ventilation,46034,34146,InvasiveVent,46034
2,ventilation,5286,2505,Tracheostomy,5286
3,ventilation,4832,2973,NonInvasiveVent,4832
4,ventilation,4746,3051,HFNC,4746
5,ventilation,197,194,None,197


---

## STEP 6: Create Suspicion of Infection Table

This table identifies suspected infection events by combining antibiotic administration with culture sampling.

**Criteria for Suspected Infection:**
- Antibiotic administered within 24 hours before to 72 hours after culture
- Both antibiotic and culture must be present

**Contains:**
- Antibiotic details (name, time, route)
- Culture information (specimen type, results)
- Suspected infection time (earliest of antibiotic or culture)

**Purpose:** Key component for Sepsis-3 definition

**Source:** `antibiotic` and `microbiologyevents` tables  
URL; mimic-iv/concepts/sepsis/suspicion_of_infection.sql

In [26]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.suspicion_of_infection` AS
WITH ab_tbl AS (
    SELECT
        ab.subject_id, ab.hadm_id, ab.stay_id
        , ab.starttime AS antibiotic_time
        , ab.stoptime AS antibiotic_stoptime
        , ab.antibiotic
        , ab.route
        , ROW_NUMBER() OVER (
            PARTITION BY ab.subject_id, ab.hadm_id
            ORDER BY ab.starttime, ab.stoptime, ab.antibiotic
        ) AS ab_id
    FROM `physionet-data.mimiciv_3_1_derived.antibiotic` ab
)
, me AS (
    SELECT
        micro.subject_id, micro.hadm_id
        , micro.chartdate, micro.charttime
        , micro.spec_type_desc, micro.org_name
        , CASE WHEN org_name IS NOT NULL AND org_name != ''
            THEN 1 ELSE 0 END AS positive_culture
    FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents` micro
)
, ab_fnl AS (
    SELECT
        ab.subject_id, ab.hadm_id, ab.stay_id
        , ab.ab_id, ab.antibiotic, ab.antibiotic_time
        , me.charttime AS culture_time
        , me.spec_type_desc AS specimen
        , me.positive_culture
        , CASE
            WHEN me.charttime IS NOT NULL THEN 1
            ELSE 0
        END AS suspected_infection
        , CASE
            WHEN ab.antibiotic_time <= me.charttime
                THEN ab.antibiotic_time
            ELSE me.charttime
        END AS suspected_infection_time
    FROM ab_tbl ab
    LEFT JOIN me
        ON ab.subject_id = me.subject_id
        AND ab.hadm_id = me.hadm_id
        AND ab.antibiotic_time >= DATETIME_SUB(me.chartdate, INTERVAL 24 HOUR)
        AND ab.antibiotic_time <= DATETIME_ADD(me.chartdate, INTERVAL 72 HOUR)
)
SELECT * FROM ab_fnl
WHERE suspected_infection = 1;

Query is running:   0%|          |

""


In [27]:
%%bigquery

SELECT
    'suspicion_of_infection' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    COUNT(DISTINCT subject_id) as unique_patients,
    SUM(positive_culture) as positive_cultures,
    COUNT(DISTINCT antibiotic) as unique_antibiotics
FROM `my-new-project-473015.my_mimiciv_derived.suspicion_of_infection`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,unique_patients,positive_cultures,unique_antibiotics
0,suspicion_of_infection,5177270,49547,78907,2052693,199


---

## STEP 7: Create SOFA Score Table

This table calculates the Sequential Organ Failure Assessment (SOFA) score for every hour of ICU stay.

**SOFA Components (6 organ systems):**
1. **Respiration** - PaO2/FiO2 ratio (with/without ventilation)
2. **Coagulation** - Platelet count
3. **Liver** - Bilirubin level
4. **Cardiovascular** - Mean arterial pressure and vasopressor use
5. **CNS** - Glasgow Coma Scale (GCS)
6. **Renal** - Creatinine and urine output

**Scoring:** Each component scored 0-4, total SOFA = sum of all components (0-24)

**Time window:** 24-hour rolling window (scores reflect worst values in past 24 hours)

**Dependencies:** vitalsign, ventilation, urine_output_rate (custom), and multiple physionet derived tables

URL: mimic-iv/concepts/score/sofa.sql

In [28]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.sofa` AS
WITH co AS (
    SELECT ih.stay_id, ie.hadm_id, hr
        , DATETIME_SUB(ih.endtime, INTERVAL '1' HOUR) AS starttime
        , ih.endtime
    FROM `physionet-data.mimiciv_3_1_derived.icustay_hourly` ih
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON ih.stay_id = ie.stay_id
)
, pafi AS (
    SELECT ie.stay_id, bg.charttime
        , CASE WHEN vd.stay_id IS NULL THEN pao2fio2ratio ELSE null
        END AS pao2fio2ratio_novent
        , CASE WHEN vd.stay_id IS NOT NULL THEN pao2fio2ratio ELSE null
        END AS pao2fio2ratio_vent
    FROM `physionet-data.mimiciv_3_1_icu.icustays` ie
    INNER JOIN `physionet-data.mimiciv_3_1_derived.bg` bg
        ON ie.subject_id = bg.subject_id
    LEFT JOIN `my-new-project-473015.my_mimiciv_derived.ventilation` vd
        ON ie.stay_id = vd.stay_id
            AND bg.charttime >= vd.starttime
            AND bg.charttime <= vd.endtime
            AND vd.ventilation_status = 'InvasiveVent'
    WHERE specimen = 'ART.'
)
, vs AS (
    SELECT co.stay_id, co.hr, MIN(vs.mbp) AS meanbp_min
    FROM co
    LEFT JOIN `my-new-project-473015.my_mimiciv_derived.vitalsign` vs
        ON co.stay_id = vs.stay_id
            AND co.starttime < vs.charttime
            AND co.endtime >= vs.charttime
    GROUP BY co.stay_id, co.hr
)
, gcs AS (
    SELECT co.stay_id, co.hr, MIN(gcs.gcs) AS gcs_min
    FROM co
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.gcs` gcs
        ON co.stay_id = gcs.stay_id
            AND co.starttime < gcs.charttime
            AND co.endtime >= gcs.charttime
    GROUP BY co.stay_id, co.hr
)
, bili AS (
    SELECT co.stay_id, co.hr, MAX(enz.bilirubin_total) AS bilirubin_max
    FROM co
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.enzyme` enz
        ON co.hadm_id = enz.hadm_id
            AND co.starttime < enz.charttime
            AND co.endtime >= enz.charttime
    GROUP BY co.stay_id, co.hr
)
, cr AS (
    SELECT co.stay_id, co.hr, MAX(chem.creatinine) AS creatinine_max
    FROM co
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.chemistry` chem
        ON co.hadm_id = chem.hadm_id
            AND co.starttime < chem.charttime
            AND co.endtime >= chem.charttime
    GROUP BY co.stay_id, co.hr
)
, plt AS (
    SELECT co.stay_id, co.hr, MIN(cbc.platelet) AS platelet_min
    FROM co
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.complete_blood_count` cbc
        ON co.hadm_id = cbc.hadm_id
            AND co.starttime < cbc.charttime
            AND co.endtime >= cbc.charttime
    GROUP BY co.stay_id, co.hr
)
, pf AS (
    SELECT co.stay_id, co.hr
        , MIN(pafi.pao2fio2ratio_novent) AS pao2fio2ratio_novent
        , MIN(pafi.pao2fio2ratio_vent) AS pao2fio2ratio_vent
    FROM co
    LEFT JOIN pafi
        ON co.stay_id = pafi.stay_id
            AND co.starttime < pafi.charttime
            AND co.endtime >= pafi.charttime
    GROUP BY co.stay_id, co.hr
)
, uo AS (
    SELECT co.stay_id, co.hr
        , MAX(
            CASE WHEN uo.uo_tm_24hr >= 22 AND uo.uo_tm_24hr <= 30
                THEN uo.urineoutput_24hr / uo.uo_tm_24hr * 24
            END) AS uo_24hr
    FROM co
    LEFT JOIN `my-new-project-473015.my_mimiciv_derived.urine_output_rate` uo
        ON co.stay_id = uo.stay_id
            AND co.starttime < uo.charttime
            AND co.endtime >= uo.charttime
    GROUP BY co.stay_id, co.hr
)
, vaso AS (
    SELECT co.stay_id, co.hr
        , MAX(epi.vaso_rate) AS rate_epinephrine
        , MAX(nor.vaso_rate) AS rate_norepinephrine
        , MAX(dop.vaso_rate) AS rate_dopamine
        , MAX(dob.vaso_rate) AS rate_dobutamine
    FROM co
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.epinephrine` epi
        ON co.stay_id = epi.stay_id
            AND co.endtime > epi.starttime AND co.endtime <= epi.endtime
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.norepinephrine` nor
        ON co.stay_id = nor.stay_id
            AND co.endtime > nor.starttime AND co.endtime <= nor.endtime
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.dopamine` dop
        ON co.stay_id = dop.stay_id
            AND co.endtime > dop.starttime AND co.endtime <= dop.endtime
    LEFT JOIN `physionet-data.mimiciv_3_1_derived.dobutamine` dob
        ON co.stay_id = dob.stay_id
            AND co.endtime > dob.starttime AND co.endtime <= dob.endtime
    WHERE epi.stay_id IS NOT NULL OR nor.stay_id IS NOT NULL
        OR dop.stay_id IS NOT NULL OR dob.stay_id IS NOT NULL
    GROUP BY co.stay_id, co.hr
)
, scorecomp AS (
    SELECT co.stay_id, co.hr, co.starttime, co.endtime
        , pf.pao2fio2ratio_novent, pf.pao2fio2ratio_vent
        , vaso.rate_epinephrine, vaso.rate_norepinephrine
        , vaso.rate_dopamine, vaso.rate_dobutamine
        , vs.meanbp_min, gcs.gcs_min, uo.uo_24hr
        , bili.bilirubin_max, cr.creatinine_max, plt.platelet_min
    FROM co
    LEFT JOIN vs ON co.stay_id = vs.stay_id AND co.hr = vs.hr
    LEFT JOIN gcs ON co.stay_id = gcs.stay_id AND co.hr = gcs.hr
    LEFT JOIN bili ON co.stay_id = bili.stay_id AND co.hr = bili.hr
    LEFT JOIN cr ON co.stay_id = cr.stay_id AND co.hr = cr.hr
    LEFT JOIN plt ON co.stay_id = plt.stay_id AND co.hr = plt.hr
    LEFT JOIN pf ON co.stay_id = pf.stay_id AND co.hr = pf.hr
    LEFT JOIN uo ON co.stay_id = uo.stay_id AND co.hr = uo.hr
    LEFT JOIN vaso ON co.stay_id = vaso.stay_id AND co.hr = vaso.hr
)
, scorecalc AS (
    SELECT scorecomp.*
        , CASE
            WHEN pao2fio2ratio_vent < 100 THEN 4
            WHEN pao2fio2ratio_vent < 200 THEN 3
            WHEN pao2fio2ratio_novent < 300 THEN 2
            WHEN pao2fio2ratio_vent < 300 THEN 2
            WHEN pao2fio2ratio_novent < 400 THEN 1
            WHEN pao2fio2ratio_vent < 400 THEN 1
            WHEN COALESCE(pao2fio2ratio_vent, pao2fio2ratio_novent) IS NULL THEN null
            ELSE 0
        END AS respiration
        , CASE
            WHEN platelet_min < 20 THEN 4
            WHEN platelet_min < 50 THEN 3
            WHEN platelet_min < 100 THEN 2
            WHEN platelet_min < 150 THEN 1
            WHEN platelet_min IS NULL THEN null
            ELSE 0
        END AS coagulation
        , CASE
            WHEN bilirubin_max >= 12.0 THEN 4
            WHEN bilirubin_max >= 6.0 THEN 3
            WHEN bilirubin_max >= 2.0 THEN 2
            WHEN bilirubin_max >= 1.2 THEN 1
            WHEN bilirubin_max IS NULL THEN null
            ELSE 0
        END AS liver
        , CASE
            WHEN rate_dopamine > 15 OR rate_epinephrine > 0.1
                OR rate_norepinephrine > 0.1 THEN 4
            WHEN rate_dopamine > 5 OR rate_epinephrine <= 0.1
                OR rate_norepinephrine <= 0.1 THEN 3
            WHEN rate_dopamine > 0 OR rate_dobutamine > 0 THEN 2
            WHEN meanbp_min < 70 THEN 1
            WHEN COALESCE(meanbp_min, rate_dopamine, rate_dobutamine,
                rate_epinephrine, rate_norepinephrine) IS NULL THEN null
            ELSE 0
        END AS cardiovascular
        , CASE
            WHEN (gcs_min >= 13 AND gcs_min <= 14) THEN 1
            WHEN (gcs_min >= 10 AND gcs_min <= 12) THEN 2
            WHEN (gcs_min >= 6 AND gcs_min <= 9) THEN 3
            WHEN gcs_min < 6 THEN 4
            WHEN gcs_min IS NULL THEN null
            ELSE 0
        END AS cns
        , CASE
            WHEN (creatinine_max >= 5.0) THEN 4
            WHEN uo_24hr < 200 THEN 4
            WHEN (creatinine_max >= 3.5 AND creatinine_max < 5.0) THEN 3
            WHEN uo_24hr < 500 THEN 3
            WHEN (creatinine_max >= 2.0 AND creatinine_max < 3.5) THEN 2
            WHEN (creatinine_max >= 1.2 AND creatinine_max < 2.0) THEN 1
            WHEN COALESCE(uo_24hr, creatinine_max) IS NULL THEN null
            ELSE 0
        END AS renal
    FROM scorecomp
)
, score_final AS (
    SELECT s.*
        , COALESCE(MAX(respiration) OVER w, 0) AS respiration_24hours
        , COALESCE(MAX(coagulation) OVER w, 0) AS coagulation_24hours
        , COALESCE(MAX(liver) OVER w, 0) AS liver_24hours
        , COALESCE(MAX(cardiovascular) OVER w, 0) AS cardiovascular_24hours
        , COALESCE(MAX(cns) OVER w, 0) AS cns_24hours
        , COALESCE(MAX(renal) OVER w, 0) AS renal_24hours
        , COALESCE(MAX(respiration) OVER w, 0)
        + COALESCE(MAX(coagulation) OVER w, 0)
        + COALESCE(MAX(liver) OVER w, 0)
        + COALESCE(MAX(cardiovascular) OVER w, 0)
        + COALESCE(MAX(cns) OVER w, 0)
        + COALESCE(MAX(renal) OVER w, 0) AS sofa_24hours
    FROM scorecalc s
    WINDOW w AS (
        PARTITION BY stay_id
        ORDER BY hr
        ROWS BETWEEN 23 PRECEDING AND 0 FOLLOWING
    )
)
SELECT * FROM score_final WHERE hr >= 0;

Query is running:   0%|          |

""


In [29]:
%%bigquery

SELECT
    'sofa' as table_name,
    COUNT(*) as row_count,
    COUNT(DISTINCT stay_id) as unique_icu_stays,
    ROUND(AVG(sofa_24hours), 2) as avg_sofa,
    MIN(sofa_24hours) as min_sofa,
    MAX(sofa_24hours) as max_sofa,
    SUM(CASE WHEN sofa_24hours >= 2 THEN 1 ELSE 0 END) as rows_with_sofa_ge_2
FROM `my-new-project-473015.my_mimiciv_derived.sofa`;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count,unique_icu_stays,avg_sofa,min_sofa,max_sofa,rows_with_sofa_ge_2
0,sofa,8215784,94437,4.16,0,23,6184366


---

## STEP 8: Create Sepsis-3 Table (Final Cohort)

This table identifies the final Sepsis-3 cohort by combining:
1. **Suspected infection** (antibiotic + culture)
2. **SOFA score ≥ 2** (organ dysfunction)

**Sepsis-3 Definition:**
- Suspected infection within 48 hours before to 24 hours after SOFA ≥ 2
- Assumes baseline SOFA = 0 before ICU admission

**Onset time:** Earliest of suspected infection time or SOFA ≥ 2 time

**Output:** One row per ICU stay with earliest Sepsis-3 event

**Dependencies:** `suspicion_of_infection` and `sofa` tables

In [30]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.sepsis3` AS
WITH sofa AS (
    SELECT stay_id, starttime, endtime
        , respiration_24hours AS respiration
        , coagulation_24hours AS coagulation
        , liver_24hours AS liver
        , cardiovascular_24hours AS cardiovascular
        , cns_24hours AS cns
        , renal_24hours AS renal
        , sofa_24hours AS sofa_score
    FROM `my-new-project-473015.my_mimiciv_derived.sofa`
    WHERE sofa_24hours >= 2
)
, s1 AS (
    SELECT
        soi.subject_id, soi.stay_id
        , soi.ab_id, soi.antibiotic, soi.antibiotic_time
        , soi.culture_time, soi.suspected_infection
        , soi.suspected_infection_time, soi.specimen
        , soi.positive_culture
        , starttime, endtime
        , respiration, coagulation, liver, cardiovascular, cns, renal
        , sofa_score
        , sofa_score >= 2 AND suspected_infection = 1 AS sepsis3
        , ROW_NUMBER() OVER (
            PARTITION BY soi.stay_id
            ORDER BY suspected_infection_time, antibiotic_time,
                     culture_time, endtime
        ) AS rn_sus
    FROM `my-new-project-473015.my_mimiciv_derived.suspicion_of_infection` AS soi
    INNER JOIN sofa
        ON soi.stay_id = sofa.stay_id
            AND sofa.endtime >= DATETIME_SUB(
                soi.suspected_infection_time, INTERVAL '48' HOUR
            )
            AND sofa.endtime <= DATETIME_ADD(
                soi.suspected_infection_time, INTERVAL '24' HOUR
            )
    WHERE soi.stay_id IS NOT NULL
)
SELECT
    subject_id, stay_id, antibiotic_time, culture_time
    , suspected_infection_time
    , endtime AS sofa_time, sofa_score
    , respiration, coagulation, liver, cardiovascular, cns, renal
    , sepsis3
FROM s1
WHERE rn_sus = 1;

Query is running:   0%|          |

""


In [31]:
%%bigquery

SELECT
    'sepsis3' as table_name,
    COUNT(DISTINCT stay_id) AS sepsis3_icu_stays,
    COUNT(DISTINCT subject_id) AS sepsis3_patients,
    ROUND(AVG(sofa_score), 2) AS avg_sofa_score,
    MIN(sofa_score) AS min_sofa,
    MAX(sofa_score) AS max_sofa,
    SUM(CASE WHEN sepsis3 = TRUE THEN 1 ELSE 0 END) as confirmed_sepsis3
FROM `my-new-project-473015.my_mimiciv_derived.sepsis3`
WHERE sepsis3 = TRUE;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,sepsis3_icu_stays,sepsis3_patients,avg_sofa_score,min_sofa,max_sofa,confirmed_sepsis3
0,sepsis3,43705,33311,3.71,2,20,43705


---

## ✅ Summary: All Tables Created Successfully

We have successfully created all 8 tables for Sepsis-3 analysis in the `my_mimiciv_derived` dataset.

**Created Tables:**
1. ✅ vitalsign
2. ✅ weight_durations
3. ✅ urine_output
4. ✅ urine_output_rate
5. ✅ ventilator_setting (prerequisite)
6. ✅ ventilation
7. ✅ suspicion_of_infection
8. ✅ sofa
9. ✅ sepsis3

Let's review the final statistics for all tables.

In [34]:
%%bigquery

SELECT
    table_id as table_name,
    TIMESTAMP_MILLIS(creation_time) as created_at,
    row_count,
    ROUND(size_bytes / 1024 / 1024, 2) as size_mb
FROM `my-new-project-473015.my_mimiciv_derived.__TABLES__`
ORDER BY table_id;

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,created_at,row_count,size_mb
0,sepsis3,2025-10-09 15:03:18.393000+00:00,43705,4.38
1,sofa,2025-10-09 15:01:46.146000+00:00,8215784,907.56
2,suspicion_of_infection,2025-10-09 14:59:28.367000+00:00,5177270,469.25
3,urine_output,2025-10-09 14:39:26.891000+00:00,4127634,94.47
4,urine_output_rate,2025-10-09 14:45:52.386000+00:00,4126485,553.54
5,ventilation,2025-10-09 14:58:06.324000+00:00,144921,5.71
6,ventilator_setting,2025-10-09 14:54:12.815000+00:00,1377514,102.48
7,vitalsign,2025-10-09 14:34:11.500000+00:00,13519533,895.50
8,weight_durations,2025-10-09 14:35:37.623000+00:00,401850,14.95


In [35]:
%%bigquery

WITH sepsis_stats AS (
    SELECT
        COUNT(DISTINCT stay_id) AS sepsis3_icu_stays,
        COUNT(DISTINCT subject_id) AS sepsis3_patients,
        ROUND(AVG(sofa_score), 2) AS avg_sofa_score,
        MIN(sofa_score) AS min_sofa,
        MAX(sofa_score) AS max_sofa
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3`
    WHERE sepsis3 = TRUE
)
, total_stats AS (
    SELECT COUNT(DISTINCT stay_id) as total_icu_stays
    FROM `physionet-data.mimiciv_3_1_icu.icustays`
)
SELECT
    s.*,
    t.total_icu_stays,
    ROUND(100.0 * s.sepsis3_icu_stays / t.total_icu_stays, 1) as sepsis3_percentage
FROM sepsis_stats s, total_stats t;

Query is running:   0%|          |

Downloading:   0%|          |

,sepsis3_icu_stays,sepsis3_patients,avg_sofa_score,min_sofa,max_sofa,total_icu_stays,sepsis3_percentage
0,43705,33311,3.71,2,20,94458,46.3
